## Overview
This chapter covers:
1. **TensorFlow Serving (TFS)**: Deploying models to production using Docker
2. **SavedModel Format**: TensorFlow's standard serialization format for models
3. **GPU Utilization**: Leveraging GPUs for faster training
4. **Distributed Training**: Training models across multiple devices and machines
5. **Model Versioning**: Managing different versions of deployed models
6. **REST and gRPC APIs**: Two ways to query deployed models

# Neural Network and Deep Learning

## **Chapter 19 – Training and Deploying TensorFlow Models at Scale**

# Setup
First, let's import a few common modules, ensure MatplotLib plots figures inline and prepare a function to save the figures. We also check that Python 3.5 or later is installed (although Python 2.x may work, it is deprecated so we strongly recommend you use Python 3 instead), as well as Scikit-Learn ≥0.20 and TensorFlow ≥2.0.


In [1]:
# Python ≥3.5 is required
import sys
assert sys.version_info >= (3, 5)

# Is this notebook running on Colab or Kaggle?
IS_COLAB = "google.colab" in sys.modules
IS_KAGGLE = "kaggle_secrets" in sys.modules

if IS_COLAB or IS_KAGGLE:
    !echo "deb http://storage.googleapis.com/tensorflow-serving-apt stable tensorflow-model-server tensorflow-model-server-universal" > /etc/apt/sources.list.d/tensorflow-serving.list
    !curl https://storage.googleapis.com/tensorflow-serving-apt/tensorflow-serving.release.pub.gpg | apt-key add -
    !apt update && apt-get install -y tensorflow-model-server
    %pip install -q -U tensorflow-serving-api

# Scikit-Learn ≥0.20 is required
import sklearn
assert sklearn.__version__ >= "0.20"

# TensorFlow ≥2.0 is required
import tensorflow as tf
from tensorflow import keras
assert tf.__version__ >= "2.0"

if not tf.config.list_physical_devices('GPU'):
    print("No GPU was detected. CNNs can be very slow without a GPU.")
    if IS_COLAB:
        print("Go to Runtime > Change runtime and select a GPU hardware accelerator.")
    if IS_KAGGLE:
        print("Go to Settings > Accelerator and select GPU.")

# Common imports
import numpy as np
import os

# to make this notebook's output stable across runs
np.random.seed(42)
tf.random.set_seed(42)

# To plot pretty figures
%matplotlib inline
import matplotlib as mpl
import matplotlib.pyplot as plt
mpl.rc('axes', labelsize=14)
mpl.rc('xtick', labelsize=12)
mpl.rc('ytick', labelsize=12)

# Where to save the figures
PROJECT_ROOT_DIR = "."
CHAPTER_ID = "deploy"
IMAGES_PATH = os.path.join(PROJECT_ROOT_DIR, "images", CHAPTER_ID)
os.makedirs(IMAGES_PATH, exist_ok=True)

def save_fig(fig_id, tight_layout=True, fig_extension="png", resolution=300):
    path = os.path.join(IMAGES_PATH, fig_id + "." + fig_extension)
    print("Saving figure", fig_id)
    if tight_layout:
        plt.tight_layout()
    plt.savefig(path, format=fig_extension, dpi=resolution)

No GPU was detected. CNNs can be very slow without a GPU.


d:\Source Code\ML\Hands-On Machine Learning with Scikit-Learn, Keras and Tensor Flow (Aurelien Geron)\.venv\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


### Environment Setup
This cell configures the Python environment by:
- **Importing TensorFlow and Keras**: Core deep learning frameworks
- **Setting random seeds**: Ensures reproducibility of results
- **GPU Detection**: Checks for available GPU devices for faster computation
- **Configuring warnings**: Suppresses unnecessary output for cleaner results

# Deploying TensorFlow models to TensorFlow Serving (TFS)
We will use the REST API or the gRPC API.

## Save/Load a `SavedModel`

## Part 1: Training and Saving Models

This section covers the foundational steps:
1. Load and prepare MNIST dataset
2. Train a simple neural network
3. Save model in SavedModel format for deployment

In [2]:
(X_train_full, y_train_full), (X_test, y_test) = keras.datasets.mnist.load_data()
X_train_full = X_train_full[..., np.newaxis].astype(np.float32) / 255.
X_test = X_test[..., np.newaxis].astype(np.float32) / 255.
X_valid, X_train = X_train_full[:5000], X_train_full[5000:]
y_valid, y_train = y_train_full[:5000], y_train_full[5000:]
X_new = X_test[:3]

In [3]:
np.random.seed(42)
tf.random.set_seed(42)

model = keras.models.Sequential([
    keras.layers.Flatten(input_shape=[28, 28, 1]),
    keras.layers.Dense(100, activation="relu"),
    keras.layers.Dense(10, activation="softmax")
])
model.compile(loss="sparse_categorical_crossentropy",
              optimizer=keras.optimizers.SGD(learning_rate=1e-2),
              metrics=["accuracy"])
model.fit(X_train, y_train, epochs=10, validation_data=(X_valid, y_valid))

d:\Source Code\ML\Hands-On Machine Learning with Scikit-Learn, Keras and Tensor Flow (Aurelien Geron)\.venv\Lib\site-packages\keras\src\layers\reshaping\flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/10
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.8339 - loss: 0.6645 - val_accuracy: 0.8996 - val_loss: 0.3653
Epoch 2/10
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.9036 - loss: 0.3441 - val_accuracy: 0.9188 - val_loss: 0.2941
Epoch 3/10
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.9170 - loss: 0.2940 - val_accuracy: 0.9280 - val_loss: 0.2598
Epoch 4/10
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.9255 - loss: 0.2638 - val_accuracy: 0.9352 - val_loss: 0.2363
Epoch 5/10
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.9319 - loss: 0.2412 - val_accuracy: 0.9406 - val_loss: 0.2177
Epoch 6/10
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.9374 - loss: 0.2230 - val_accuracy: 0.9462 - val_loss: 0.2027
Epoch 7/10
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.9416 - loss: 0.2077 - val_accuracy: 0.9494 - val_loss: 0.1899
Epoch 8/10
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.9455 - loss: 0.1945 - 

### Training a Simple MNIST Model
This cell demonstrates training a basic neural network that will later be deployed:
- **MNIST Dataset**: Classic handwritten digit dataset (28x28 grayscale images)
- **Model Architecture**: Sequential model with:
  - Flatten layer to convert 2D images to 1D vectors
  - Dense hidden layer with 100 neurons and ReLU activation
  - Output layer with 10 neurons (one per digit class) and softmax activation
- **Training**: Uses sparse categorical crossentropy loss and SGD optimizer
- **Purpose**: This model will be saved and deployed using TensorFlow Serving

In [4]:
np.round(model.predict(X_new), 2)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step


array([[0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 1.  , 0.  , 0.  ],
       [0.  , 0.  , 0.99, 0.01, 0.  , 0.  , 0.  , 0.  , 0.  , 0.  ],
       [0.  , 0.98, 0.01, 0.  , 0.  , 0.  , 0.  , 0.01, 0.  , 0.  ]],
      dtype=float32)

In [5]:
model_version = "0001"
model_name = "my_mnist_model"
model_path = os.path.join(model_name, model_version)
model_path

'my_mnist_model\\0001'

In [6]:
import shutil

shutil.rmtree(model_name)

FileNotFoundError: [WinError 3] The system cannot find the path specified: 'my_mnist_model'

In [8]:
# Using model.save() instead of tf.saved_model.save() for compatibility
import os
os.makedirs(model_path, exist_ok=True)
model.export(model_path)

INFO:tensorflow:Assets written to: my_mnist_model\0001\assets


INFO:tensorflow:Assets written to: my_mnist_model\0001\assets


Saved artifact at 'my_mnist_model\0001'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 28, 28, 1), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 10), dtype=tf.float32, name=None)
Captures:
  1407065961296: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1407065962256: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1407065961680: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1407065962064: TensorSpec(shape=(), dtype=tf.resource, name=None)


### Saving Models in SavedModel Format
**SavedModel** is TensorFlow's universal serialization format for deploying models:

**Key Concepts:**
- **Complete Model Package**: Includes architecture, weights, computation graph, and metadata
- **Version Management**: Models are saved with version numbers (0001, 0002, etc.)
- **TF Serving Compatible**: This format is required for TensorFlow Serving deployment
- **Language Agnostic**: Can be loaded in Python, C++, Java, Go, and other languages

**Directory Structure:**
```
my_mnist_model/
├── 0001/              # Version 1
│   ├── saved_model.pb # Computation graph
│   ├── variables/     # Model weights
│   └── assets/        # Additional files
└── 0002/              # Version 2 (new trained model)
```

**`model.export()` vs `tf.saved_model.save()`:**
- Modern Keras 3.x uses `model.export()` for better compatibility
- Older TF 2.x used `tf.saved_model.save()`
- Both create the same SavedModel format

In [9]:
for root, dirs, files in os.walk(model_name):
    indent = '    ' * root.count(os.sep)
    print('{}{}/'.format(indent, os.path.basename(root)))
    for filename in files:
        print('{}{}'.format(indent + '    ', filename))

my_mnist_model/
    0001/
        fingerprint.pb
        saved_model.pb
        assets/
        variables/
            variables.data-00000-of-00001
            variables.index


### Inspecting SavedModel Structure
Exploring the SavedModel directory contents:
- **`saved_model.pb`**: Contains the computation graph serialized as a protocol buffer
- **`variables/`**: Directory storing all model weights and parameters
- **`assets/`**: Optional directory for additional files (e.g., vocabulary files for NLP)
- **Version folders (0001, 0002)**: Each version is self-contained for rollback capabilities

In [10]:
!saved_model_cli show --dir {model_path}

In [11]:
!saved_model_cli show --dir {model_path} --tag_set serve

In [12]:
!saved_model_cli show --dir {model_path} --tag_set serve \
                      --signature_def serving_default

In [13]:
!saved_model_cli show --dir {model_path} --all

Let's write the new instances to a `npy` file so we can pass them easily to our model:

In [14]:
np.save("my_mnist_tests.npy", X_new)

### Loading SavedModel in Python
**`tf.saved_model.load()`** reconstructs the model from disk:
- Returns a TensorFlow function (not a Keras model)
- Used primarily for inference/predictions
- Cannot be retrained directly (use `tf.keras.models.load_model()` for training)
- Useful for verifying model integrity before deployment

In [17]:
# Build the model first by making a prediction to define input shape
_ = model.predict(X_new[:1])
# Now we can get the input name
input_name = model.layers[0].name + "_input"
input_name

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step


'flatten_input'

And now let's use `saved_model_cli` to make predictions for the instances we just saved:

In [18]:
!saved_model_cli run --dir {model_path} --tag_set serve \
                     --signature_def serving_default    \
                     --inputs {input_name}=my_mnist_tests.npy

### Creating Prediction Request
**Format for TF Serving:**
- Input must match the model's expected signature
- Data serialized to JSON for REST API
- NumPy arrays converted to nested Python lists
- Response contains predictions in same format

In [19]:
np.round([[1.1347984e-04, 1.5187356e-07, 9.7032893e-04, 2.7640699e-03, 3.7826971e-06,
           7.6876910e-05, 3.9140293e-08, 9.9559116e-01, 5.3502394e-05, 4.2665208e-04],
          [8.2443521e-04, 3.5493889e-05, 9.8826385e-01, 7.0466995e-03, 1.2957400e-07,
           2.3389691e-04, 2.5639210e-03, 9.5886099e-10, 1.0314899e-03, 8.7952529e-08],
          [4.4693781e-05, 9.7028232e-01, 9.0526715e-03, 2.2641101e-03, 4.8766597e-04,
           2.8800720e-03, 2.2714981e-03, 8.3753867e-03, 4.0439744e-03, 2.9759688e-04]], 2)

array([[0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 1.  , 0.  , 0.  ],
       [0.  , 0.  , 0.99, 0.01, 0.  , 0.  , 0.  , 0.  , 0.  , 0.  ],
       [0.  , 0.97, 0.01, 0.  , 0.  , 0.  , 0.  , 0.01, 0.  , 0.  ]])

## TensorFlow Serving

Install [Docker](https://docs.docker.com/install/) if you don't have it already. Then run:

```bash
docker pull tensorflow/serving

export ML_PATH=$HOME/ml # or wherever this project is
docker run -it --rm -p 8500:8500 -p 8501:8501 \
   -v "$ML_PATH/my_mnist_model:/models/my_mnist_model" \
   -e MODEL_NAME=my_mnist_model \
   tensorflow/serving
```
Once you are finished using it, press Ctrl-C to shut down the server.

## TensorFlow Serving (TFS)

**TensorFlow Serving** is a production-ready model serving system:

### Key Features:
1. **High Performance**: Optimized for low-latency inference
2. **Model Versioning**: Serve multiple model versions simultaneously
3. **Hot Swapping**: Update models without downtime
4. **Auto-scaling**: Handle varying load efficiently
5. **Monitoring**: Built-in metrics and logging

### Deployment Options:
- **Docker**: Easiest way to run TFS (recommended)
- **Native Installation**: Direct installation on Linux servers
- **Kubernetes**: Orchestration for production environments

### API Types:
- **REST API**: Simple HTTP requests, easy to test with curl/Postman
- **gRPC API**: Higher performance, uses protocol buffers

**Note**: The following cells require TensorFlow Serving to be running. Install with:
```bash
docker run -p 8500:8500 -p 8501:8501 --mount type=bind,source=/path/to/models,target=/models/my_mnist_model -e MODEL_NAME=my_mnist_model -t tensorflow/serving
```

Alternatively, if `tensorflow_model_server` is installed (e.g., if you are running this notebook in Colab), then the following 3 cells will start the server:

In [17]:
os.environ["MODEL_DIR"] = os.path.split(os.path.abspath(model_path))[0]

### Making Predictions via REST API
**REST API Request Format:**
```json
{
  "signature_name": "serving_default",
  "instances": [[...], [...], ...]
}
```

- **Endpoint**: `http://localhost:8501/v1/models/my_mnist_model:predict`
- **Method**: POST
- **Content-Type**: application/json
- **Port 8501**: REST API (HTTP)
- **Port 8500**: gRPC API (binary protocol)

**Advantages of REST:**
- Easy to test with curl or web browsers
- Works with any HTTP client
- Human-readable JSON format

In [18]:
%%bash --bg
nohup tensorflow_model_server \
     --rest_api_port=8501 \
     --model_name=my_mnist_model \
     --model_base_path="${MODEL_DIR}" >server.log 2>&1

In [19]:
!tail server.log

2021-02-16 22:33:09.323538: I external/org_tensorflow/tensorflow/cc/saved_model/reader.cc:93] Reading SavedModel debug info (if present) from: /models/my_mnist_model/0001
2021-02-16 22:33:09.323642: I external/org_tensorflow/tensorflow/core/platform/cpu_feature_guard.cc:142] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2021-02-16 22:33:09.360572: I external/org_tensorflow/tensorflow/cc/saved_model/loader.cc:206] Restoring SavedModel bundle.
2021-02-16 22:33:09.361764: I external/org_tensorflow/tensorflow/core/platform/profile_utils/cpu_utils.cc:112] CPU Frequency: 2200000000 Hz
2021-02-16 22:33:09.387713: I external/org_tensorflow/tensorflow/cc/saved_model/loader.cc:190] Running initialization op on SavedModel bundle at path: /models/my_mnist_model/0001
2021-02-16 22:33:09.

In [20]:
import json

input_data_json = json.dumps({
    "signature_name": "serving_default",
    "instances": X_new.tolist(),
})

In [21]:
repr(input_data_json)[:1500] + "..."

'\'{"signature_name": "serving_default", "instances": [[[[0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0]], [[0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0]], [[0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0]], [[0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0]], [[0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0

Now let's use TensorFlow Serving's REST API to make predictions:

In [22]:
import requests

SERVER_URL = 'http://localhost:8501/v1/models/my_mnist_model:predict'
response = requests.post(SERVER_URL, data=input_data_json)
response.raise_for_status() # raise an exception in case of error
response = response.json()

In [23]:
response.keys()

dict_keys(['predictions'])

In [24]:
y_proba = np.array(response["predictions"])
y_proba.round(2)

array([[0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 1.  , 0.  , 0.  ],
       [0.  , 0.  , 0.99, 0.01, 0.  , 0.  , 0.  , 0.  , 0.  , 0.  ],
       [0.  , 0.97, 0.01, 0.  , 0.  , 0.  , 0.  , 0.01, 0.  , 0.  ]])

### Using the gRPC API

In [25]:
from tensorflow_serving.apis.predict_pb2 import PredictRequest

request = PredictRequest()
request.model_spec.name = model_name
request.model_spec.signature_name = "serving_default"
input_name = model.input_names[0]
request.inputs[input_name].CopyFrom(tf.make_tensor_proto(X_new))

### Making Predictions via gRPC API
**gRPC (Google Remote Procedure Call)** offers better performance:

**Advantages over REST:**
- **Faster**: Binary protocol buffers instead of JSON
- **Efficient**: Smaller message size
- **Streaming**: Supports bidirectional streaming
- **Type Safety**: Strong typing with .proto definitions

**Trade-offs:**
- More complex to set up
- Requires tensorflow-serving-api package
- Less human-readable than JSON

**Use Cases:**
- High-throughput production systems
- When millisecond latency matters
- Mobile/IoT devices with limited bandwidth

In [26]:
import grpc
from tensorflow_serving.apis import prediction_service_pb2_grpc

channel = grpc.insecure_channel('localhost:8500')
predict_service = prediction_service_pb2_grpc.PredictionServiceStub(channel)
response = predict_service.Predict(request, timeout=10.0)

In [27]:
response

outputs {
  key: "dense_1"
  value {
    dtype: DT_FLOAT
    tensor_shape {
      dim {
        size: 3
      }
      dim {
        size: 10
      }
    }
    float_val: 0.00011425172124290839
    float_val: 1.513665068841874e-07
    float_val: 0.0009818424005061388
    float_val: 0.0027773496694862843
    float_val: 3.758880893656169e-06
    float_val: 7.6266449468676e-05
    float_val: 3.9139514740327286e-08
    float_val: 0.995561957359314
    float_val: 5.344580131350085e-05
    float_val: 0.00043088122038170695
    float_val: 0.0008194865076802671
    float_val: 3.5498320357874036e-05
    float_val: 0.9882420897483826
    float_val: 0.00705744931474328
    float_val: 1.2937064752804872e-07
    float_val: 0.00023402832448482513
    float_val: 0.0025743397418409586
    float_val: 9.668431610876382e-10
    float_val: 0.0010369382798671722
    float_val: 8.833576004008137e-08
    float_val: 4.441547571332194e-05
    float_val: 0.970328688621521
    float_val: 0.009044423699378967
    

Convert the response to a tensor:

In [28]:
output_name = model.output_names[0]
outputs_proto = response.outputs[output_name]
y_proba = tf.make_ndarray(outputs_proto)
y_proba.round(2)

array([[0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 1.  , 0.  , 0.  ],
       [0.  , 0.  , 0.99, 0.01, 0.  , 0.  , 0.  , 0.  , 0.  , 0.  ],
       [0.  , 0.97, 0.01, 0.  , 0.  , 0.  , 0.  , 0.01, 0.  , 0.  ]],
      dtype=float32)

Or to a NumPy array if your client does not include the TensorFlow library:

In [29]:
output_name = model.output_names[0]
outputs_proto = response.outputs[output_name]
shape = [dim.size for dim in outputs_proto.tensor_shape.dim]
y_proba = np.array(outputs_proto.float_val).reshape(shape)
y_proba.round(2)

array([[0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 1.  , 0.  , 0.  ],
       [0.  , 0.  , 0.99, 0.01, 0.  , 0.  , 0.  , 0.  , 0.  , 0.  ],
       [0.  , 0.97, 0.01, 0.  , 0.  , 0.  , 0.  , 0.01, 0.  , 0.  ]])

## Deploying a new model version

In [20]:
np.random.seed(42)
tf.random.set_seed(42)

model = keras.models.Sequential([
    keras.layers.Flatten(input_shape=[28, 28, 1]),
    keras.layers.Dense(50, activation="relu"),
    keras.layers.Dense(50, activation="relu"),
    keras.layers.Dense(10, activation="softmax")
])
model.compile(loss="sparse_categorical_crossentropy",
              optimizer=keras.optimizers.SGD(learning_rate=1e-2),
              metrics=["accuracy"])
history = model.fit(X_train, y_train, epochs=10, validation_data=(X_valid, y_valid))

Epoch 1/10


d:\Source Code\ML\Hands-On Machine Learning with Scikit-Learn, Keras and Tensor Flow (Aurelien Geron)\.venv\Lib\site-packages\keras\src\layers\reshaping\flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


1719/1719 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.7926 - loss: 0.7693 - val_accuracy: 0.9020 - val_loss: 0.3592
Epoch 2/10
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 2s 973us/step - accuracy: 0.9047 - loss: 0.3381 - val_accuracy: 0.9218 - val_loss: 0.2875
Epoch 3/10
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 2s 988us/step - accuracy: 0.9167 - loss: 0.2898 - val_accuracy: 0.9278 - val_loss: 0.2547
Epoch 4/10
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 2s 986us/step - accuracy: 0.9252 - loss: 0.2588 - val_accuracy: 0.9364 - val_loss: 0.2303
Epoch 5/10
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.9326 - loss: 0.2336 - val_accuracy: 0.9410 - val_loss: 0.2102
Epoch 6/10
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.9382 - loss: 0.2125 - val_accuracy: 0.9462 - val_loss: 0.1937
Epoch 7/10
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 2s 969us/step - accuracy: 0.9439 - loss: 0.1947 - val_accuracy: 0.9522 - val_loss: 0.1792
Epoch 8/10
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.9483 - loss: 0.1792 - val

### Deploying a New Model Version
**Model Versioning Benefits:**
- **A/B Testing**: Compare performance of different models
- **Rollback**: Quickly revert to previous version if issues arise
- **Gradual Rollout**: Route percentage of traffic to new version
- **Zero Downtime**: TF Serving automatically loads new versions

**How it works:**
1. Save new model with higher version number (e.g., 0002)
2. TF Serving detects new version automatically
3. Both versions remain available
4. Can specify version in API request or use latest by default

In [21]:
model_version = "0002"
model_name = "my_mnist_model"
model_path = os.path.join(model_name, model_version)
model_path

'my_mnist_model\\0002'

In [23]:
# Using model.export() instead of tf.saved_model.save() for compatibility
import os
os.makedirs(model_path, exist_ok=True)
model.export(model_path)

INFO:tensorflow:Assets written to: my_mnist_model\0002\assets


INFO:tensorflow:Assets written to: my_mnist_model\0002\assets


Saved artifact at 'my_mnist_model\0002'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 28, 28, 1), dtype=tf.float32, name='keras_tensor_4')
Output Type:
  TensorSpec(shape=(None, 10), dtype=tf.float32, name=None)
Captures:
  1407177276688: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1407177277072: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1407177276496: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1407177274000: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1407177276880: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1407177274192: TensorSpec(shape=(), dtype=tf.resource, name=None)


In [24]:
for root, dirs, files in os.walk(model_name):
    indent = '    ' * root.count(os.sep)
    print('{}{}/'.format(indent, os.path.basename(root)))
    for filename in files:
        print('{}{}'.format(indent + '    ', filename))

my_mnist_model/
    0001/
        fingerprint.pb
        saved_model.pb
        assets/
        variables/
            variables.data-00000-of-00001
            variables.index
    0002/
        fingerprint.pb
        saved_model.pb
        assets/
        variables/
            variables.data-00000-of-00001
            variables.index


**Warning**: You may need to wait a minute before the new model is loaded by TensorFlow Serving.

In [34]:
import requests

SERVER_URL = 'http://localhost:8501/v1/models/my_mnist_model:predict'
            
response = requests.post(SERVER_URL, data=input_data_json)
response.raise_for_status()
response = response.json()

### Querying Model Metadata
**Model Signature Information:**
- **Inputs**: Expected tensor names, shapes, and dtypes
- **Outputs**: Output tensor specifications
- **Signature Name**: Usually "serving_default"

This information is crucial for:
- Building correct API requests
- Debugging input/output mismatches
- Understanding model interface

In [35]:
response.keys()

dict_keys(['predictions'])

In [36]:
y_proba = np.array(response["predictions"])
y_proba.round(2)

array([[0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 1.  , 0.  , 0.  ],
       [0.  , 0.  , 0.99, 0.01, 0.  , 0.  , 0.  , 0.  , 0.  , 0.  ],
       [0.  , 0.99, 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  ]])

# Deploy the model to Google Cloud AI Platform

Follow the instructions in the book to deploy the model to Google Cloud AI Platform, download the service account's private key and save it to the `my_service_account_private_key.json` in the project directory. Also, update the `project_id`:

In [37]:
project_id = "onyx-smoke-242003"

In [38]:
import googleapiclient.discovery

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "my_service_account_private_key.json"
model_id = "my_mnist_model"
model_path = "projects/{}/models/{}".format(project_id, model_id)
model_path += "/versions/v0001/" # if you want to run a specific version
ml_resource = googleapiclient.discovery.build("ml", "v1").projects()

In [39]:
def predict(X):
    input_data_json = {"signature_name": "serving_default",
                       "instances": X.tolist()}
    request = ml_resource.predict(name=model_path, body=input_data_json)
    response = request.execute()
    if "error" in response:
        raise RuntimeError(response["error"])
    return np.array([pred[output_name] for pred in response["predictions"]])

In [40]:
Y_probas = predict(X_new)
np.round(Y_probas, 2)

array([[0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 1.  , 0.  , 0.  ],
       [0.  , 0.  , 0.99, 0.01, 0.  , 0.  , 0.  , 0.  , 0.  , 0.  ],
       [0.  , 0.99, 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  ]])

# Using GPUs

**Note**: `tf.test.is_gpu_available()` is deprecated. Instead, please use `tf.config.list_physical_devices('GPU')`.

In [25]:
#tf.test.is_gpu_available() # deprecated
tf.config.list_physical_devices('GPU')

[]

## GPU Acceleration

### Why Use GPUs for Deep Learning?
- **Parallelism**: GPUs have thousands of cores vs CPUs with ~10-20 cores
- **Speed**: 10-100x faster training for large neural networks
- **Matrix Operations**: Optimized for the linear algebra operations in deep learning

### GPU Detection
TensorFlow automatically detects and uses available GPUs. Key considerations:
- **CUDA**: NVIDIA's parallel computing platform (required for GPU support)
- **cuDNN**: NVIDIA's deep learning library (optimizes common operations)
- **Memory Management**: GPUs have limited VRAM; batch size must fit in memory

In [26]:
tf.test.gpu_device_name()

''

In [27]:
tf.test.is_built_with_cuda()

False

In [28]:
from tensorflow.python.client.device_lib import list_local_devices

devices = list_local_devices()
devices

[name: "/device:CPU:0"
 device_type: "CPU"
 memory_limit: 268435456
 locality {
 }
 incarnation: 14886916994659768401
 xla_global_id: -1]

# Distributed Training

In [29]:
keras.backend.clear_session()
tf.random.set_seed(42)
np.random.seed(42)

## Distributed Training with TensorFlow

### Distribution Strategies
TensorFlow provides several strategies for distributed training:

1. **MirroredStrategy** (used here):
   - Data Parallelism: Each device gets a full copy of the model
   - Synchronous training: All replicas train on different data batches
   - Gradients are averaged across all devices
   - Best for: Multiple GPUs on a single machine

2. **MultiWorkerMirroredStrategy**:
   - Similar to MirroredStrategy but across multiple machines
   - Uses collective communication for gradient aggregation
   - Best for: Large-scale training across clusters

3. **ParameterServerStrategy**:
   - Asynchronous training
   - Separate machines for parameters and workers
   - Best for: Very large models that don't fit on one machine

4. **CentralStorageStrategy**:
   - All variables stored on CPU
   - Computation on GPUs
   - Best for: Small models with fast GPU operations

In [30]:
def create_model():
    return keras.models.Sequential([
        keras.layers.Conv2D(filters=64, kernel_size=7, activation="relu",
                            padding="same", input_shape=[28, 28, 1]),
        keras.layers.MaxPooling2D(pool_size=2),
        keras.layers.Conv2D(filters=128, kernel_size=3, activation="relu",
                            padding="same"), 
        keras.layers.Conv2D(filters=128, kernel_size=3, activation="relu",
                            padding="same"),
        keras.layers.MaxPooling2D(pool_size=2),
        keras.layers.Flatten(),
        keras.layers.Dense(units=64, activation='relu'),
        keras.layers.Dropout(0.5),
        keras.layers.Dense(units=10, activation='softmax'),
    ])

### Creating Distributed Dataset
**Why Distribute Data?**
When training across multiple devices, data must be distributed efficiently:

**`experimental_distribute_dataset()`:**
- Automatically shards dataset across replicas
- Each device gets different portion of data
- Ensures no duplicate processing
- Handles synchronization automatically

**Best Practices:**
- Use `tf.data.Dataset` API for input pipeline
- Apply transformations before distribution
- Use `.cache()` and `.prefetch()` for performance

In [31]:
batch_size = 100
model = create_model()
model.compile(loss="sparse_categorical_crossentropy",
              optimizer=keras.optimizers.SGD(learning_rate=1e-2),
              metrics=["accuracy"])
model.fit(X_train, y_train, epochs=10,
          validation_data=(X_valid, y_valid), batch_size=batch_size)

Epoch 1/10


d:\Source Code\ML\Hands-On Machine Learning with Scikit-Learn, Keras and Tensor Flow (Aurelien Geron)\.venv\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


550/550 ━━━━━━━━━━━━━━━━━━━━ 47s 84ms/step - accuracy: 0.5953 - loss: 1.2865 - val_accuracy: 0.9148 - val_loss: 0.3229
Epoch 2/10
550/550 ━━━━━━━━━━━━━━━━━━━━ 46s 84ms/step - accuracy: 0.8660 - loss: 0.4437 - val_accuracy: 0.9522 - val_loss: 0.1767
Epoch 3/10
550/550 ━━━━━━━━━━━━━━━━━━━━ 47s 86ms/step - accuracy: 0.9077 - loss: 0.3045 - val_accuracy: 0.9654 - val_loss: 0.1275
Epoch 4/10
550/550 ━━━━━━━━━━━━━━━━━━━━ 48s 88ms/step - accuracy: 0.9316 - loss: 0.2372 - val_accuracy: 0.9710 - val_loss: 0.1038
Epoch 5/10
550/550 ━━━━━━━━━━━━━━━━━━━━ 49s 89ms/step - accuracy: 0.9417 - loss: 0.2036 - val_accuracy: 0.9744 - val_loss: 0.0887
Epoch 6/10
550/550 ━━━━━━━━━━━━━━━━━━━━ 49s 89ms/step - accuracy: 0.9480 - loss: 0.1808 - val_accuracy: 0.9768 - val_loss: 0.0800
Epoch 7/10
550/550 ━━━━━━━━━━━━━━━━━━━━ 46s 83ms/step - accuracy: 0.9525 - loss: 0.1629 - val_accuracy: 0.9782 - val_loss: 0.0734
Epoch 8/10
550/550 ━━━━━━━━━━━━━━━━━━━━ 53s 96ms/step - accuracy: 0.9562 - loss: 0.1498 - val_accurac

### Training Model with Distribution Strategy
**Seamless Distributed Training:**
Once model is created within `distribution.scope()`, training works identically to single-device training:

```python
with distribution.scope():
    model = create_model()  # Distributed automatically
model.fit(dataset)  # TensorFlow handles all distribution
```

**What TensorFlow handles automatically:**
- Gradient computation on each replica
- Gradient aggregation (AllReduce)
- Weight synchronization
- Loss scaling for numerical stability

In [32]:
keras.backend.clear_session()
tf.random.set_seed(42)
np.random.seed(42)

distribution = tf.distribute.MirroredStrategy()

# Change the default all-reduce algorithm:
#distribution = tf.distribute.MirroredStrategy(
#    cross_device_ops=tf.distribute.HierarchicalCopyAllReduce())

# Specify the list of GPUs to use:
#distribution = tf.distribute.MirroredStrategy(devices=["/gpu:0", "/gpu:1"])

# Use the central storage strategy instead:
#distribution = tf.distribute.experimental.CentralStorageStrategy()

#if IS_COLAB and "COLAB_TPU_ADDR" in os.environ:
#  tpu_address = "grpc://" + os.environ["COLAB_TPU_ADDR"]
#else:
#  tpu_address = ""
#resolver = tf.distribute.cluster_resolver.TPUClusterResolver(tpu_address)
#tf.config.experimental_connect_to_cluster(resolver)
#tf.tpu.experimental.initialize_tpu_system(resolver)
#distribution = tf.distribute.experimental.TPUStrategy(resolver)

with distribution.scope():
    model = create_model()
    model.compile(loss="sparse_categorical_crossentropy",
                  optimizer=keras.optimizers.SGD(learning_rate=1e-2),
                  metrics=["accuracy"])

INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:CPU:0',)


INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:CPU:0',)


### Training with MirroredStrategy
**Data Parallelism with Synchronous Training:**

**How it works:**
1. **Model Replication**: Full model copy on each GPU
2. **Batch Splitting**: Input batch divided across GPUs
3. **Forward Pass**: Each GPU processes its portion independently
4. **Gradient Computation**: Each GPU calculates gradients
5. **AllReduce**: Gradients averaged across all GPUs using efficient algorithms
6. **Weight Update**: Synchronized update ensures all replicas stay identical

**Key Code Pattern:**
```python
with distribution.scope():
    model = create_model()  # Model created in strategy scope
    model.compile(...)
model.fit(...)  # TensorFlow handles distribution automatically
```

**Batch Size Considerations:**
- Global batch size = per-replica batch size × number of replicas
- Larger global batch may require learning rate adjustment

In [33]:
batch_size = 100 # must be divisible by the number of workers
model.fit(X_train, y_train, epochs=10,
          validation_data=(X_valid, y_valid), batch_size=batch_size)

Epoch 1/10
550/550 ━━━━━━━━━━━━━━━━━━━━ 47s 86ms/step - accuracy: 0.5948 - loss: 1.2929 - val_accuracy: 0.9070 - val_loss: 0.3485
Epoch 2/10
550/550 ━━━━━━━━━━━━━━━━━━━━ 47s 85ms/step - accuracy: 0.8658 - loss: 0.4507 - val_accuracy: 0.9474 - val_loss: 0.1908
Epoch 3/10
550/550 ━━━━━━━━━━━━━━━━━━━━ 50s 90ms/step - accuracy: 0.9126 - loss: 0.2985 - val_accuracy: 0.9626 - val_loss: 0.1304
Epoch 4/10
550/550 ━━━━━━━━━━━━━━━━━━━━ 47s 86ms/step - accuracy: 0.9336 - loss: 0.2311 - val_accuracy: 0.9712 - val_loss: 0.1051
Epoch 5/10
550/550 ━━━━━━━━━━━━━━━━━━━━ 48s 88ms/step - accuracy: 0.9443 - loss: 0.1935 - val_accuracy: 0.9740 - val_loss: 0.0925
Epoch 6/10
550/550 ━━━━━━━━━━━━━━━━━━━━ 50s 90ms/step - accuracy: 0.9511 - loss: 0.1690 - val_accuracy: 0.9786 - val_loss: 0.0809
Epoch 7/10
550/550 ━━━━━━━━━━━━━━━━━━━━ 47s 85ms/step - accuracy: 0.9541 - loss: 0.1570 - val_accuracy: 0.9782 - val_loss: 0.0746
Epoch 8/10
550/550 ━━━━━━━━━━━━━━━━━━━━ 50s 90ms/step - accuracy: 0.9587 - loss: 0.1418 - 

In [34]:
model.predict(X_new)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 161ms/step


array([[2.42049367e-08, 1.44764599e-07, 8.44026786e-08, 1.26314220e-07,
        5.74654591e-09, 1.59025237e-09, 2.09894810e-10, 9.99997735e-01,
        2.32076172e-08, 1.82614929e-06],
       [1.33688741e-06, 4.49026584e-06, 9.99990463e-01, 1.32071548e-07,
        3.07205178e-10, 5.03587276e-11, 7.24675928e-08, 2.80027534e-09,
        3.47957075e-06, 5.26830446e-12],
       [2.59347689e-06, 9.99882102e-01, 3.16290311e-06, 2.72623947e-06,
        1.16142583e-05, 1.07894084e-05, 4.81958959e-05, 1.85824119e-05,
        1.47377705e-05, 5.34017681e-06]], dtype=float32)

Custom training loop:

In [35]:
keras.backend.clear_session()
tf.random.set_seed(42)
np.random.seed(42)

K = keras.backend

distribution = tf.distribute.MirroredStrategy()

with distribution.scope():
    model = create_model()
    optimizer = keras.optimizers.SGD()

with distribution.scope():
    dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train)).repeat().batch(batch_size)
    input_iterator = distribution.make_dataset_iterator(dataset)
    
@tf.function
def train_step():
    def step_fn(inputs):
        X, y = inputs
        with tf.GradientTape() as tape:
            Y_proba = model(X)
            loss = K.sum(keras.losses.sparse_categorical_crossentropy(y, Y_proba)) / batch_size

        grads = tape.gradient(loss, model.trainable_variables)
        optimizer.apply_gradients(zip(grads, model.trainable_variables))
        return loss

    per_replica_losses = distribution.experimental_run(step_fn, input_iterator)
    mean_loss = distribution.reduce(tf.distribute.ReduceOp.SUM,
                                    per_replica_losses, axis=None)
    return mean_loss

n_epochs = 10
with distribution.scope():
    input_iterator.initialize()
    for epoch in range(n_epochs):
        print("Epoch {}/{}".format(epoch + 1, n_epochs))
        for iteration in range(len(X_train) // batch_size):
            print("\rLoss: {:.3f}".format(train_step().numpy()), end="")
        print()

INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:CPU:0',)


INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:CPU:0',)


Instructions for updating:
Use the iterator's `initializer` property instead.


Instructions for updating:
Use the iterator's `initializer` property instead.


Epoch 1/10
Instructions for updating:
use run() instead


Instructions for updating:
use run() instead


Loss: 0.408
Epoch 2/10
Loss: 0.312
Epoch 3/10
Loss: 0.293
Epoch 4/10
Loss: 0.293
Epoch 5/10
Loss: 0.291
Epoch 6/10
Loss: 0.283
Epoch 7/10
Loss: 0.278
Epoch 8/10
Loss: 0.268
Epoch 9/10
Loss: 0.260
Epoch 10/10
Loss: 0.254


### Custom Training Loop with Distribution
**When to use custom training loops:**
- Need fine-grained control over training process
- Implementing custom gradient manipulation
- Research experiments with novel training algorithms
- Complex multi-loss scenarios

**Key Components:**
1. **`@tf.function`**: Compiles Python function to TensorFlow graph for speed
2. **`distribution.run()`**: Executes function on all replicas
3. **`GradientTape`**: Records operations for automatic differentiation
4. **`reduce_mean()`**: Aggregates per-replica losses

**Performance Benefits:**
- Graph execution (via @tf.function) is 10-50x faster than eager mode
- Distribution handled automatically within the decorated function

## Training across multiple servers

A TensorFlow cluster is a group of TensorFlow processes running in parallel, usually on different machines, and talking to each other to complete some work, for example training or executing a neural network. Each TF process in the cluster is called a "task" (or a "TF server"). It has an IP address, a port, and a type (also called its role or its job). The type can be `"worker"`, `"chief"`, `"ps"` (parameter server) or `"evaluator"`:
* Each **worker** performs computations, usually on a machine with one or more GPUs.
* The **chief** performs computations as well, but it also handles extra work such as writing TensorBoard logs or saving checkpoints. There is a single chief in a cluster, typically the first worker (i.e., worker #0).
* A **parameter server** (ps) only keeps track of variable values, it is usually on a CPU-only machine.
* The **evaluator** obviously takes care of evaluation. There is usually a single evaluator in a cluster.

The set of tasks that share the same type is often called a "job". For example, the "worker" job is the set of all workers.

To start a TensorFlow cluster, you must first define it. This means specifying all the tasks (IP address, TCP port, and type). For example, the following cluster specification defines a cluster with 3 tasks (2 workers and 1 parameter server). It's a dictionary with one key per job, and the values are lists of task addresses:

In [36]:
cluster_spec = {
    "worker": [
        "machine-a.example.com:2222",  # /job:worker/task:0
        "machine-b.example.com:2222"   # /job:worker/task:1
    ],
    "ps": ["machine-c.example.com:2222"] # /job:ps/task:0
}

## Multi-Worker Training

### Training Across Multiple Machines
**Cluster Configuration:**
- **Workers**: Machines that train the model
- **Chief**: Special worker (typically worker 0) that coordinates
- **Parameter Servers** (optional): Store model variables

**TF_CONFIG Environment Variable:**
```json
{
  "cluster": {
    "worker": ["host1:port1", "host2:port2", "host3:port3"]
  },
  "task": {"type": "worker", "index": 0}
}
```

**Communication Protocols:**
- **All-Reduce**: Efficient gradient aggregation (ring or tree topology)
- **Collective Ops**: TensorFlow's optimized multi-machine operations
- **gRPC**: Default protocol for worker communication

Every task in the cluster may communicate with every other task in the server, so make sure to configure your firewall to authorize all communications between these machines on these ports (it's usually simpler if you use the same port on every machine).

When a task is started, it needs to be told which one it is: its type and index (the task index is also called the task id). A common way to specify everything at once (both the cluster spec and the current task's type and id) is to set the `TF_CONFIG` environment variable before starting the program. It must be a JSON-encoded dictionary containing a cluster specification (under the `"cluster"` key), and the type and index of the task to start (under the `"task"` key). For example, the following `TF_CONFIG` environment variable defines the same cluster as above, with 2 workers and 1 parameter server, and specifies that the task to start is worker #1:

In [37]:
import os
import json

os.environ["TF_CONFIG"] = json.dumps({
    "cluster": cluster_spec,
    "task": {"type": "worker", "index": 1}
})
os.environ["TF_CONFIG"]

'{"cluster": {"worker": ["machine-a.example.com:2222", "machine-b.example.com:2222"], "ps": ["machine-c.example.com:2222"]}, "task": {"type": "worker", "index": 1}}'

### TFConfigClusterResolver
**Purpose**: Automatically reads TF_CONFIG environment variable

**What it provides:**
- Cluster specification (worker addresses and ports)
- Current task information (type and index)
- Simplifies cluster configuration management

**Use Case**: When running distributed training, each worker needs to know:
1. Where are the other workers? (cluster spec)
2. Who am I? (task type and index)

This resolver eliminates manual parsing of environment variables.

Some platforms (e.g., Google Cloud ML Engine) automatically set this environment variable for you.

TensorFlow's `TFConfigClusterResolver` class reads the cluster configuration from this environment variable:

In [38]:
import tensorflow as tf

resolver = tf.distribute.cluster_resolver.TFConfigClusterResolver()
resolver.cluster_spec()

ClusterSpec({'ps': ['machine-c.example.com:2222'], 'worker': ['machine-a.example.com:2222', 'machine-b.example.com:2222']})

In [39]:
resolver.task_type

'worker'

In [40]:
resolver.task_id

1

Now let's run a simpler cluster with just two worker tasks, both running on the local machine. We will use the `MultiWorkerMirroredStrategy` to train a model across these two tasks.

The first step is to write the training code. As this code will be used to run both workers, each in its own process, we write this code to a separate Python file, `my_mnist_multiworker_task.py`. The code is relatively straightforward, but there are a couple important things to note:
* We create the `MultiWorkerMirroredStrategy` before doing anything else with TensorFlow.
* Only one of the workers will take care of logging to TensorBoard and saving checkpoints. As mentioned earlier, this worker is called the *chief*, and by convention it is usually worker #0.

In [41]:
%%writefile my_mnist_multiworker_task.py

import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
import time

# At the beginning of the program
distribution = tf.distribute.MultiWorkerMirroredStrategy()

resolver = tf.distribute.cluster_resolver.TFConfigClusterResolver()
print("Starting task {}{}".format(resolver.task_type, resolver.task_id))

# Only worker #0 will write checkpoints and log to TensorBoard
if resolver.task_id == 0:
    root_logdir = os.path.join(os.curdir, "my_mnist_multiworker_logs")
    run_id = time.strftime("run_%Y_%m_%d-%H_%M_%S")
    run_dir = os.path.join(root_logdir, run_id)
    callbacks = [
        keras.callbacks.TensorBoard(run_dir),
        keras.callbacks.ModelCheckpoint("my_mnist_multiworker_model.h5",
                                        save_best_only=True),
    ]
else:
    callbacks = []

# Load and prepare the MNIST dataset
(X_train_full, y_train_full), (X_test, y_test) = keras.datasets.mnist.load_data()
X_train_full = X_train_full[..., np.newaxis] / 255.
X_valid, X_train = X_train_full[:5000], X_train_full[5000:]
y_valid, y_train = y_train_full[:5000], y_train_full[5000:]

with distribution.scope():
    model = keras.models.Sequential([
        keras.layers.Conv2D(filters=64, kernel_size=7, activation="relu",
                            padding="same", input_shape=[28, 28, 1]),
        keras.layers.MaxPooling2D(pool_size=2),
        keras.layers.Conv2D(filters=128, kernel_size=3, activation="relu",
                            padding="same"), 
        keras.layers.Conv2D(filters=128, kernel_size=3, activation="relu",
                            padding="same"),
        keras.layers.MaxPooling2D(pool_size=2),
        keras.layers.Flatten(),
        keras.layers.Dense(units=64, activation='relu'),
        keras.layers.Dropout(0.5),
        keras.layers.Dense(units=10, activation='softmax'),
    ])
    model.compile(loss="sparse_categorical_crossentropy",
                  optimizer=keras.optimizers.SGD(learning_rate=1e-2),
                  metrics=["accuracy"])

model.fit(X_train, y_train, validation_data=(X_valid, y_valid),
          epochs=10, callbacks=callbacks)

Writing my_mnist_multiworker_task.py


### Creating Training Script for Multi-Worker
**This cell generates a standalone Python script** that can be executed on each worker machine.

**Key Modifications for Multi-Worker:**
1. **Different file paths**: Each worker needs unique paths to avoid conflicts
2. **MultiWorkerMirroredStrategy**: Replaces MirroredStrategy
3. **Callbacks**: 
   - `BackupAndRestore`: Saves checkpoints for fault tolerance
   - Only chief worker should save final model

**Fault Tolerance:**
- Workers can fail and restart from last checkpoint
- Training continues without losing progress
- Essential for long-running distributed jobs

**Execution:**
Each worker runs this script with different TF_CONFIG settings.

In a real world application, there would typically be a single worker per machine, but in this example we're running both workers on the same machine, so they will both try to use all the available GPU RAM (if this machine has a GPU), and this will likely lead to an Out-Of-Memory (OOM) error. To avoid this, we could use the `CUDA_VISIBLE_DEVICES` environment variable to assign a different GPU to each worker. Alternatively, we can simply disable GPU support, like this:

In [42]:
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

### Running Workers as Subprocesses
**Simulation of Multi-Machine Setup:**
This cell demonstrates running multiple workers on a single machine by:
1. Starting each worker as a separate subprocess
2. Each worker gets unique TF_CONFIG with its task index
3. Workers communicate via localhost on different ports

**In Production:**
- Each worker runs on a separate physical/virtual machine
- Uses actual network addresses instead of localhost
- Requires proper network configuration and firewall rules
- May use Kubernetes or other orchestration platforms

**Note**: This is a demonstration; real distributed training runs on separate machines.

We are now ready to start both workers, each in its own process, using Python's `subprocess` module. Before we start each process, we need to set the `TF_CONFIG` environment variable appropriately, changing only the task index:

In [59]:
import subprocess

cluster_spec = {"worker": ["127.0.0.1:9901", "127.0.0.1:9902"]}

for index, worker_address in enumerate(cluster_spec["worker"]):
    os.environ["TF_CONFIG"] = json.dumps({
        "cluster": cluster_spec,
        "task": {"type": "worker", "index": index}
    })
    subprocess.Popen("python my_mnist_multiworker_task.py", shell=True)

That's it! Our TensorFlow cluster is now running, but we can't see it in this notebook because it's running in separate processes (but if you are running this notebook in Jupyter, you can see the worker logs in Jupyter's server logs).

Since the chief (worker #0) is writing to TensorBoard, we use TensorBoard to view the training progress. Run the following cell, then click on the settings button (i.e., the gear icon) in the TensorBoard interface and check the "Reload data" box to make TensorBoard automatically refresh every 30s. Once the first epoch of training is finished (which may take a few minutes), and once TensorBoard refreshes, the SCALARS tab will appear. Click on this tab to view the progress of the model's training and validation accuracy.

In [60]:
%load_ext tensorboard
%tensorboard --logdir=./my_mnist_multiworker_logs --port=6006

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


## TensorBoard for Monitoring

**TensorBoard** is TensorFlow's visualization toolkit:

### Key Features:
1. **Training Metrics**: Loss and accuracy curves
2. **Graph Visualization**: See model architecture and computation graph
3. **Histograms**: Weight and activation distributions
4. **Profiler**: Performance bottleneck analysis
5. **Hyperparameter Tuning**: Compare different configurations

### Usage:
```python
tensorboard_cb = keras.callbacks.TensorBoard(log_dir)
model.fit(..., callbacks=[tensorboard_cb])
```

Then run: `tensorboard --logdir=./my_logs --port=6006`

**Benefits:**
- Real-time monitoring during training
- Debug training issues (vanishing/exploding gradients)
- Compare multiple experiments
- Share results with team via web interface

That's it! Once training is over, the best checkpoint of the model will be available in the `my_mnist_multiworker_model.h5` file. You can load it using `keras.models.load_model()` and use it for predictions, as usual:

In [44]:
from tensorflow import keras
import os

# Check if multiworker model exists, otherwise use the regular model
if os.path.exists("my_mnist_multiworker_model.h5"):
    model = keras.models.load_model("my_mnist_multiworker_model.h5")
    Y_pred = model.predict(X_new)
    np.argmax(Y_pred, axis=-1)
else:
    print("Multiworker model not found. Skipping this cell.")
    print("To run this, execute cells 89-90 first to train the multiworker model.")

Multiworker model not found. Skipping this cell.
To run this, execute cells 89-90 first to train the multiworker model.
